In [ ]:
COMPANY = "pinecone"
BOARD = "ashby"

In [ ]:
import json, db
from collections import defaultdict

bronze_db = db.open_bronze(BOARD)
silver_db = db.open_silver()

rows = bronze_db.execute(
    "SELECT * FROM snapshots WHERE company=? ORDER BY source_date",
    (COMPANY,)
).fetchall()
print(f"Bronze records for {COMPANY}: {len(rows)}")

by_job = defaultdict(dict)
for row in rows:
    by_job[row["job_id"]][row["page_type"]] = row

print(f"Unique jobs: {len(by_job)}")

In [ ]:
%run classify.ipynb

import html2text
import re
import db as _db

def _md(html_text):
    if not html_text:
        return ""
    h = html2text.HTML2Text()
    h.body_width = 0
    h.ignore_images = True
    h.ignore_links = True
    return h.handle(html_text).strip()

def _title_from_html(html_text):
    if not html_text:
        return ""
    import re as _re
    m = _re.search(r"<title>([^<]+)</title>", html_text, re.IGNORECASE)
    return m.group(1).strip() if m else ""

def merge_to_silver(job_id, page_rows):
    api = page_rows.get("api_board")
    app = page_rows.get("application")
    page = page_rows.get("job_page")
    source_row = api or app or page
    if source_row is None:
        return None

    record = {
        "company": source_row["company"],
        "board": source_row["board"],
        "job_id": job_id,
        "source_date": source_row["source_date"],
        "bronze_id": source_row["id"],
    }

    if api:
        try:
            job = json.loads(api["content"] or "{}")
        except Exception as e:
            print(f"WARNING: json.loads failed for job_id={job_id}: {e}")
            job = {}
        record["title"] = job.get("title", "")
        loc = job.get("location")
        if isinstance(loc, dict):
            record["location"] = loc.get("name", "")
        elif isinstance(loc, list):
            record["location"] = " | ".join(l.get("name", "") for l in loc if isinstance(l, dict))
        else:
            record["location"] = loc or ""
        record["url"] = job.get("jobUrl") or job.get("absolute_url", "")
        record["department_raw"] = (
            job.get("department") or job.get("team") or
            (job.get("departments") or [{}])[0].get("name", "")
        )

    if api and not record.get("salary_min"):
        content_html = job.get("content", "")
        if content_html:
            import html as _html
            content_html = _html.unescape(content_html)
            sal_block = _db.extract_salary_block_from_html(content_html)
            if sal_block:
                parsed = _db.parse_salary_text(sal_block)
                if parsed.salary_min:
                    record["salary_min"] = int(parsed.salary_min)
                    record["salary_max"] = int(parsed.salary_max) if parsed.salary_max else None
                    record["currency"] = parsed.currency or ""
                    record["salary_unit"] = parsed.salary_unit or ""
                    record["salary_text"] = parsed.salary_text or ""

    if app:
        try:
            jld = json.loads(app["content"] or "{}")
        except Exception as e:
            print(f"WARNING: json.loads failed for application job_id={job_id}: {e}")
            jld = {}
        if jld.get("salary_min"):
            record["salary_min"] = int(jld["salary_min"])
            record["salary_max"] = int(jld["salary_max"]) if jld.get("salary_max") else None
            record["currency"] = jld.get("currency", "")
            record["salary_unit"] = jld.get("salary_unit", "")
            record["salary_text"] = jld.get("salary_text", "")
        if not record.get("title"):
            record["title"] = jld.get("title", "")
        if not record.get("location"):
            record["location"] = jld.get("location", "")

    if page:
        content = page["content"] or ""
        record["description_md"] = _md(content)
        if not record.get("salary_min"):
            sal_block = _db.extract_salary_block_from_html(content)
            parsed = _db.parse_salary_text(sal_block) if sal_block else _db.SalaryParseResult("", None, None, None, None)
            if parsed.salary_min:
                record["salary_min"] = int(parsed.salary_min)
                record["salary_max"] = int(parsed.salary_max) if parsed.salary_max else None
                record["currency"] = parsed.currency or ""
                record["salary_unit"] = parsed.salary_unit or ""
                record["salary_text"] = parsed.salary_text or ""
        if not record.get("title"):
            record["title"] = _title_from_html(content)

    title = record.get("title", "")
    record["seniority"] = classify_seniority(title)
    record["department"] = normalize_department(record.get("department_raw", ""))
    record["work_mode"] = classify_work_mode(record.get("location", ""))
    record["yoe"] = extract_yoe(record.get("description_md", "") or "")
    return record

In [ ]:
processed = upserted = rejected = 0

for job_id, page_rows in by_job.items():
    record = merge_to_silver(job_id, page_rows)
    if record is None:
        silver_db.execute(
            "INSERT INTO silver_rejected (company, job_id, source_date, rejection_reason, rejected_at) VALUES (?,?,?,?,?)",
            (COMPANY, job_id, None, "merge_to_silver returned None — no usable bronze rows",
             __import__("datetime").datetime.now(__import__("datetime").timezone.utc).isoformat())
        )
        silver_db.commit()
        rejected += 1
        continue
    processed += 1
    ok = db.upsert_job(silver_db, record)
    upserted += ok
    rejected += not ok

db.log_run(silver_db, COMPANY, BOARD, processed, upserted, rejected)
print(f"Processed: {processed}  Upserted: {upserted}  Rejected: {rejected}")